## This will converts raw text into training samples that GPT can learn from.

In [1]:
import torch
from tokenizers import Tokenizer

BATCH_SIZE = 16

BLOCK_SIZE = 128

In [2]:
# Load Tokenizer
tokenizer = Tokenizer.from_file(
    "tokenizer.json"
)

print("Tokenizer Loaded")

Tokenizer Loaded


In [3]:
# Load Dataset
with open(
    "tiny_shakespeare.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()

print(
    "Dataset Length:",
    len(text)
)

Dataset Length: 1115394


In [4]:
# convert text to token
encoded = tokenizer.encode(
    text
)

token_ids = encoded.ids

print(
    token_ids[:20]
)

[407, 765, 12, 1975, 116, 2271, 424, 1915, 8, 395, 81, 365, 10, 882, 12, 2030, 8, 365, 10, 407]


In [5]:
# convert to tensor
data = torch.tensor(
    token_ids,
    dtype=torch.long
)

print(data.shape)

torch.Size([289150])


In [6]:
# Train Validation Split
n = int(
    0.9 * len(data)
)

train_data = data[:n]

val_data = data[n:]

print(
    len(train_data)
)

print(
    len(val_data)
)

260235
28915


In [7]:
# Batch Generator
def get_batch(
    split
):
    # Select dataset
    source = (
        train_data
        if split == "train"
        else val_data
    )
    # Generate random starting positions
    ix = torch.randint(

        len(source)
        - BLOCK_SIZE
        - 1,

        (BATCH_SIZE,)
    )
    # Build input batch
    x = torch.stack(

        [

            source[
                i:i+BLOCK_SIZE
            ]

            for i in ix

        ]

    )
    # Build target batch
    y = torch.stack(

        [

            source[
                i+1:
                i+BLOCK_SIZE+1
            ]

            for i in ix

        ]

    )

    return x,y

In [8]:
# GPU Support 
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [10]:
# Move Batch to GPU
xb,yb = get_batch(
    "train"
)

xb = xb.to(device)

yb = yb.to(device)

print(
    xb.device
)

cuda:0


### What I have done here
tiny_shakespeare.txt -> Tokenizer -> Token IDs -> Train / Validation Split -> Random Context Windows -> Input Batch (x) -> Target Batch (y)